**Working of Python UDF on Cluster**
- Spark application assume is running on Cluster with 1 Driver and 2 Nodes.
- Once we write this UDF to process our logic, Spark copies this UDF to individual worker nodes.
- Once this is available spark spins up the Python Process and once this process is ready data is serialized in the format that Python can understand, Data is Processed Row by Row
- Once data is processed results are reverted back to JVM Process.
- Then JVM reports back to driver with the results.
- Python Process resides in the same node where JVM resides, if there are many UDFs written in an application and python process grows in size and exceeds memory limits we will run out of memory on the Node.
- **`Cons of UDFs`** :
     - `Data needs to be Serialized/Deserialized`
     - `Python Process has to be spin up.`
     - `Processing data row by row by python process.`
- **`Solution`** :
     - `Write UDF in Java or Scala once written register this UDF`, 
     - `Since we register the UDF already present in JVM , We dont need to spin up Python Process.`
     - We can now reuse the Java/Scala UDF within python code itself.

#### UDFs, 3 Types :
1. Scalar Python UDF
2. Pandas vectorized UDF
3. UDTF

In [0]:
emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string,hire_date string"
emp_df = spark.read.format("csv").option("header","true").schema(emp_schema).load("/Volumes/easewithdata/spark_db/data/emp.csv")

In [0]:
def bonus(salary):
    return int(salary) * 0.1

In [0]:
# Register as UDF
from pyspark.sql.functions import udf

bonus_udf = udf(bonus)

In [0]:
import pyspark.sql.functions as F

In [0]:
emp_df.withColumn("Bonus", bonus_udf(F.col("salary"))).display()

In [0]:
import time
def bonus_test(salary):
    time.sleep(10)
    return int(salary) * 0.1

In [0]:
bonus_udf_test = udf(bonus_test)

In [0]:
emp_df.withColumn("Bonus", bonus_udf_test(F.col("salary"))).display()